# Property-Market Context — Data Audit and Feature Engineering

This notebook audits and prepares the property-market contextual data for
the season-aware real-estate investment-screening framework.

The property-market data will be treated as contextual information available
at each Airbnb prediction origin. Temporal alignment will be implemented
using the pre-specified information-lag rule before any property-market
variables are integrated with the modelling dataset.

Raw source files are preserved unchanged.

In [ ]:
# =============================================================================
# STEP 22.1 — PROPERTY-MARKET RAW FILE INVENTORY
# Purpose:
#   1. Confirm the raw property-market directory exists.
#   2. Inventory all raw files currently stored in the directory.
#   3. Confirm that the expected MCERT XLSX and MBIE Rental Bond CSV are present.
#
# IMPORTANT:
#   - This step does NOT modify any raw file.
#   - This step does NOT read or clean the dataset contents.
# =============================================================================

from pathlib import Path
import pandas as pd

print("=" * 100)
print("STEP 22.1 — PROPERTY-MARKET RAW FILE INVENTORY")
print("=" * 100)

# -----------------------------------------------------------------------------
# 1. Project paths
# -----------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")
RAW_DIR = PROJECT_ROOT / "01_raw" / "property_market"

print("\nProject root:")
print(PROJECT_ROOT)

print("\nProperty-market raw directory:")
print(RAW_DIR)

print("\nDirectory exists:")
print(RAW_DIR.exists())

# -----------------------------------------------------------------------------
# 2. Inventory files
# -----------------------------------------------------------------------------

if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"Property-market raw directory does not exist:\n{RAW_DIR}"
    )

files = sorted(
    [p for p in RAW_DIR.iterdir() if p.is_file()],
    key=lambda x: x.name.lower()
)

print(f"\nNumber of files found: {len(files)}")

inventory = []

for p in files:
    inventory.append(
        {
            "file_name": p.name,
            "extension": p.suffix.lower(),
            "size_mb": round(p.stat().st_size / (1024 ** 2), 3),
            "full_path": str(p)
        }
    )

inventory_df = pd.DataFrame(inventory)

print("\nRAW PROPERTY-MARKET FILE INVENTORY")
print("-" * 100)

if len(inventory_df) == 0:
    print("No files found.")
else:
    print(
        inventory_df[
            ["file_name", "extension", "size_mb", "full_path"]
        ].to_string(index=False)
    )

# -----------------------------------------------------------------------------
# 3. Basic expected-source checks
# -----------------------------------------------------------------------------

xlsx_files = [p for p in files if p.suffix.lower() == ".xlsx"]
csv_files = [p for p in files if p.suffix.lower() == ".csv"]

print("\n" + "=" * 100)
print("EXPECTED SOURCE CHECK")
print("=" * 100)

print(f"\nXLSX files found : {len(xlsx_files)}")
for p in xlsx_files:
    print(f"  - {p.name}")

print(f"\nCSV files found  : {len(csv_files)}")
for p in csv_files:
    print(f"  - {p.name}")

# -----------------------------------------------------------------------------
# 4. Identify candidate source files by filename
# -----------------------------------------------------------------------------

mcert_candidates = [
    p for p in xlsx_files
    if "property" in p.name.lower()
    and "sales" in p.name.lower()
]

rental_candidates = [
    p for p in csv_files
    if (
        "tenancy" in p.name.lower()
        or "tla" in p.name.lower()
        or "rental" in p.name.lower()
        or "bond" in p.name.lower()
    )
]

print("\nMCERT Property & Sales candidate(s):")
if mcert_candidates:
    for p in mcert_candidates:
        print(f"  - {p.name}")
else:
    print("  NONE FOUND")

print("\nMBIE Rental Bond TA candidate(s):")
if rental_candidates:
    for p in rental_candidates:
        print(f"  - {p.name}")
else:
    print("  NONE FOUND")

# -----------------------------------------------------------------------------
# 5. Final Step 22.1 status
# -----------------------------------------------------------------------------

basic_pass = (
    RAW_DIR.exists()
    and len(mcert_candidates) == 1
    and len(rental_candidates) == 1
)

print("\n" + "=" * 100)

if basic_pass:
    print("STEP 22.1 STATUS: PASS")
    print("Expected raw sources detected:")
    print(f"  MCERT Property & Sales : {mcert_candidates[0].name}")
    print(f"  MBIE Rental Bond TA    : {rental_candidates[0].name}")
else:
    print("STEP 22.1 STATUS: CHECK REQUIRED")
    print("Expected exactly:")
    print("  - 1 MCERT Property & Sales XLSX")
    print("  - 1 MBIE Rental Bond TA CSV")

print("=" * 100)

In [ ]:
# =============================================================================
# STEP 22.2 — MBIE RENTAL BOND RAW STRUCTURAL AUDIT
#
# Purpose:
#   1. Read the raw MBIE/Tenancy Services TA-level monthly CSV.
#   2. Inspect its exact schema and dimensions.
#   3. Identify geography and time fields.
#   4. Inspect temporal coverage and uniqueness.
#   5. Inspect missingness without modifying the raw data.
#
# IMPORTANT:
#   - RAW DATA ARE READ ONLY.
#   - No cleaning.
#   - No imputation.
#   - No feature engineering.
#   - No TA crosswalk yet.
#   - No T1-T10 alignment yet.
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 110)
print("STEP 22.2 — MBIE RENTAL BOND RAW STRUCTURAL AUDIT")
print("=" * 110)

# -----------------------------------------------------------------------------
# 1. Paths
# -----------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")
RAW_DIR = PROJECT_ROOT / "01_raw" / "property_market"

RENTAL_FILE = RAW_DIR / "detailed-monthly-tla-tenancy-september.csv"

print("\nRaw file:")
print(RENTAL_FILE)

print("\nFile exists:")
print(RENTAL_FILE.exists())

if not RENTAL_FILE.exists():
    raise FileNotFoundError(f"Rental Bond file not found:\n{RENTAL_FILE}")

# -----------------------------------------------------------------------------
# 2. Read raw CSV
# -----------------------------------------------------------------------------

df = pd.read_csv(
    RENTAL_FILE,
    low_memory=False
)

print("\n" + "=" * 110)
print("BASIC STRUCTURE")
print("=" * 110)

print(f"\nRows    : {len(df):,}")
print(f"Columns : {df.shape[1]:,}")

# -----------------------------------------------------------------------------
# 3. Exact column names
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("COLUMN NAMES")
print("=" * 110)

for i, col in enumerate(df.columns, start=1):
    print(f"{i:>3}. {repr(col)}")

# -----------------------------------------------------------------------------
# 4. Data types
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("DATA TYPES")
print("=" * 110)

print(df.dtypes.to_string())

# -----------------------------------------------------------------------------
# 5. First 10 rows
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("FIRST 10 ROWS")
print("=" * 110)

print(df.head(10).to_string(index=False))

# -----------------------------------------------------------------------------
# 6. Missingness
# -----------------------------------------------------------------------------

missing = pd.DataFrame({
    "column": df.columns,
    "missing_n": [df[c].isna().sum() for c in df.columns],
    "missing_percent": [
        df[c].isna().mean() * 100 for c in df.columns
    ]
})

missing = missing.sort_values(
    ["missing_percent", "column"],
    ascending=[False, True]
).reset_index(drop=True)

print("\n" + "=" * 110)
print("MISSINGNESS BY COLUMN")
print("=" * 110)

print(
    missing.to_string(
        index=False,
        formatters={
            "missing_percent": lambda x: f"{x:.3f}%"
        }
    )
)

# -----------------------------------------------------------------------------
# 7. Search for likely geography and time columns
# -----------------------------------------------------------------------------

lower_cols = {c: c.lower().strip() for c in df.columns}

geo_keywords = [
    "territorial",
    "authority",
    "tla",
    "area",
    "district",
    "location",
    "region"
]

time_keywords = [
    "date",
    "month",
    "period",
    "year",
    "quarter"
]

geo_candidates = [
    c for c in df.columns
    if any(k in lower_cols[c] for k in geo_keywords)
]

time_candidates = [
    c for c in df.columns
    if any(k in lower_cols[c] for k in time_keywords)
]

print("\n" + "=" * 110)
print("LIKELY GEOGRAPHY / TIME COLUMNS")
print("=" * 110)

print("\nGeography candidates:")
for c in geo_candidates:
    print(f"  - {c}")

print("\nTime candidates:")
for c in time_candidates:
    print(f"  - {c}")

# -----------------------------------------------------------------------------
# 8. Unique-value audit for likely geography fields
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("GEOGRAPHY CANDIDATE AUDIT")
print("=" * 110)

if geo_candidates:
    for c in geo_candidates:
        n_unique = df[c].nunique(dropna=True)

        print(f"\nColumn: {c}")
        print(f"Unique non-missing values: {n_unique:,}")

        # Only print values if reasonably small
        if n_unique <= 100:
            vals = sorted(
                df[c].dropna().astype(str).unique().tolist()
            )

            print("Values:")
            for v in vals:
                print(f"  - {v}")
        else:
            print("First 20 unique values:")
            vals = (
                df[c]
                .dropna()
                .astype(str)
                .drop_duplicates()
                .head(20)
                .tolist()
            )

            for v in vals:
                print(f"  - {v}")
else:
    print("\nNo obvious geography column detected automatically.")

# -----------------------------------------------------------------------------
# 9. Preliminary time-field audit
#
# We do NOT alter the raw dataframe.
# We only attempt temporary parsing for inspection.
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("TIME CANDIDATE AUDIT")
print("=" * 110)

if time_candidates:
    for c in time_candidates:

        print(f"\nColumn: {c}")
        print(f"Raw dtype: {df[c].dtype}")
        print(f"Unique non-missing values: {df[c].nunique(dropna=True):,}")

        sample_values = (
            df[c]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .head(15)
            .tolist()
        )

        print("Sample raw values:")
        for v in sample_values:
            print(f"  - {v}")

        # Temporary parse only
        parsed = pd.to_datetime(df[c], errors="coerce")

        parse_success = parsed.notna().sum()
        parse_rate = parse_success / len(df) * 100 if len(df) else np.nan

        print(f"Datetime parse success: {parse_success:,}/{len(df):,} "
              f"({parse_rate:.2f}%)")

        if parse_success > 0:
            print(f"Parsed minimum: {parsed.min()}")
            print(f"Parsed maximum: {parsed.max()}")

else:
    print("\nNo obvious time column detected automatically.")

# -----------------------------------------------------------------------------
# 10. Exact duplicate-row audit
# -----------------------------------------------------------------------------

exact_duplicate_rows = df.duplicated().sum()

print("\n" + "=" * 110)
print("EXACT DUPLICATE ROW AUDIT")
print("=" * 110)

print(f"\nExact duplicate rows: {exact_duplicate_rows:,}")

# -----------------------------------------------------------------------------
# 11. Basic numeric-column audit
# -----------------------------------------------------------------------------

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print("\n" + "=" * 110)
print("NUMERIC COLUMN AUDIT")
print("=" * 110)

print(f"\nNumber of numeric columns: {len(numeric_cols)}")

for c in numeric_cols:
    s = df[c]

    print(
        f"{c}: "
        f"n={s.notna().sum():,}, "
        f"missing={s.isna().sum():,}, "
        f"min={s.min() if s.notna().any() else np.nan}, "
        f"max={s.max() if s.notna().any() else np.nan}"
    )

# -----------------------------------------------------------------------------
# 12. Final status
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)

if len(df) > 0 and df.shape[1] > 0:
    print("STEP 22.2 STATUS: RAW STRUCTURE READ SUCCESSFULLY")
    print("No cleaning or feature engineering has been performed.")
    print("Further QC is required before this dataset can be used.")
else:
    print("STEP 22.2 STATUS: CHECK REQUIRED")

print("=" * 110)

In [ ]:
# =============================================================================
# STEP 22.3 — MBIE RENTAL BOND GEOGRAPHY + TEMPORAL COVERAGE AUDIT
#
# Purpose:
#   1. Parse TimeFrame explicitly as day/month/year.
#   2. Audit location_id = -99 and -1.
#   3. Determine actual geography coverage.
#   4. Determine exact monthly coverage.
#   5. Test location × month structural completeness.
#
# IMPORTANT:
#   - Audit only.
#   - No cleaning.
#   - No feature engineering.
#   - No imputation.
#   - No TA2025 crosswalk yet.
# =============================================================================

import pandas as pd
import numpy as np

print("=" * 110)
print("STEP 22.3 — MBIE RENTAL BOND GEOGRAPHY + TEMPORAL COVERAGE AUDIT")
print("=" * 110)

# -------------------------------------------------------------------------
# 1. Parse TimeFrame explicitly
# -------------------------------------------------------------------------

parsed_date = pd.to_datetime(
    df["TimeFrame"],
    format="%d/%m/%Y",
    errors="coerce"
)

print("\nTEMPORAL PARSING")
print("-" * 110)

print(f"Rows                         : {len(df):,}")
print(f"Successfully parsed dates    : {parsed_date.notna().sum():,}")
print(f"Failed date parses           : {parsed_date.isna().sum():,}")

if parsed_date.notna().any():
    print(f"Minimum date                 : {parsed_date.min().date()}")
    print(f"Maximum date                 : {parsed_date.max().date()}")
    print(f"Unique monthly dates         : {parsed_date.nunique():,}")

# Check whether all dates are first day of month
non_first_day = (parsed_date.dropna().dt.day != 1).sum()

print(f"Dates not on day 1           : {non_first_day:,}")

# -------------------------------------------------------------------------
# 2. Expected continuous monthly range
# -------------------------------------------------------------------------

if parsed_date.notna().any():

    expected_months = pd.date_range(
        parsed_date.min(),
        parsed_date.max(),
        freq="MS"
    )

    observed_months = pd.DatetimeIndex(
        sorted(parsed_date.dropna().unique())
    )

    missing_months = expected_months.difference(observed_months)

    print("\nMONTHLY CONTINUITY")
    print("-" * 110)

    print(f"Expected months              : {len(expected_months):,}")
    print(f"Observed unique months       : {len(observed_months):,}")
    print(f"Missing months               : {len(missing_months):,}")

    if len(missing_months) > 0:
        print("\nMissing month(s):")
        for x in missing_months:
            print(f"  - {x.strftime('%Y-%m')}")
    else:
        print("Monthly sequence is continuous.")

# -------------------------------------------------------------------------
# 3. location_id / location mapping
# -------------------------------------------------------------------------

print("\n" + "=" * 110)
print("LOCATION ID ↔ LOCATION MAPPING")
print("=" * 110)

location_map = (
    df[["location_id", "location"]]
    .drop_duplicates()
    .sort_values(["location_id", "location"], na_position="first")
)

print(f"\nUnique ID-name combinations: {len(location_map):,}")

print("\nComplete mapping:")
print(location_map.to_string(index=False))

# -------------------------------------------------------------------------
# 4. Special IDs
# -------------------------------------------------------------------------

print("\n" + "=" * 110)
print("SPECIAL LOCATION IDs")
print("=" * 110)

for special_id in [-99, -1]:

    tmp = df.loc[
        df["location_id"] == special_id
    ].copy()

    print(f"\nlocation_id = {special_id}")
    print(f"Rows                  : {len(tmp):,}")
    print(f"Unique location names : {tmp['location'].nunique(dropna=False):,}")

    print("Location values:")
    print(tmp["location"].value_counts(dropna=False).to_string())

    if len(tmp) > 0:
        tmp_dates = pd.to_datetime(
            tmp["TimeFrame"],
            format="%d/%m/%Y",
            errors="coerce"
        )

        print(f"Minimum date           : {tmp_dates.min()}")
        print(f"Maximum date           : {tmp_dates.max()}")
        print(f"Unique months          : {tmp_dates.nunique():,}")

        print("\nFirst 5 rows:")
        print(tmp.head(5).to_string(index=False))

# -------------------------------------------------------------------------
# 5. Regular positive location IDs
# -------------------------------------------------------------------------

regular = df.loc[df["location_id"] > 0].copy()

regular["_parsed_date"] = pd.to_datetime(
    regular["TimeFrame"],
    format="%d/%m/%Y",
    errors="coerce"
)

print("\n" + "=" * 110)
print("POSITIVE LOCATION-ID COVERAGE")
print("=" * 110)

print(f"\nRows with location_id > 0    : {len(regular):,}")
print(f"Unique positive IDs          : {regular['location_id'].nunique():,}")
print(f"Unique non-missing names     : {regular['location'].nunique(dropna=True):,}")
print(f"Missing location names       : {regular['location'].isna().sum():,}")

# -------------------------------------------------------------------------
# 6. Number of rows per month
# -------------------------------------------------------------------------

rows_per_month = (
    df.assign(_parsed_date=parsed_date)
      .groupby("_parsed_date", dropna=False)
      .size()
)

print("\n" + "=" * 110)
print("ROWS PER MONTH — ALL RECORDS")
print("=" * 110)

print(f"\nMinimum rows/month : {rows_per_month.min():,}")
print(f"Maximum rows/month : {rows_per_month.max():,}")
print(f"Unique row counts  : {sorted(rows_per_month.unique().tolist())}")

print("\nFrequency of monthly row counts:")
print(rows_per_month.value_counts().sort_index().to_string())

# -------------------------------------------------------------------------
# 7. Positive IDs per month
# -------------------------------------------------------------------------

positive_ids_per_month = (
    regular.groupby("_parsed_date")["location_id"]
    .nunique()
)

print("\n" + "=" * 110)
print("POSITIVE LOCATION IDs PER MONTH")
print("=" * 110)

print(f"\nMinimum positive IDs/month : {positive_ids_per_month.min():,}")
print(f"Maximum positive IDs/month : {positive_ids_per_month.max():,}")
print(
    "Unique counts             : "
    f"{sorted(positive_ids_per_month.unique().tolist())}"
)

print("\nFrequency:")
print(
    positive_ids_per_month
    .value_counts()
    .sort_index()
    .to_string()
)

# -------------------------------------------------------------------------
# 8. Duplicate month × location_id keys
# -------------------------------------------------------------------------

key_test = df.assign(_parsed_date=parsed_date)

duplicate_keys = key_test.duplicated(
    subset=["_parsed_date", "location_id"],
    keep=False
)

n_duplicate_rows = duplicate_keys.sum()

print("\n" + "=" * 110)
print("MONTH × LOCATION_ID KEY AUDIT")
print("=" * 110)

print(f"\nRows involved in duplicate month-location keys: {n_duplicate_rows:,}")

if n_duplicate_rows > 0:
    print("\nFirst duplicate-key records:")
    print(
        key_test.loc[
            duplicate_keys,
            [
                "_parsed_date",
                "location_id",
                "location",
                "LodgedBonds",
                "ActiveBonds",
                "MedianRent"
            ]
        ]
        .sort_values(["_parsed_date", "location_id"])
        .head(30)
        .to_string(index=False)
    )

# -------------------------------------------------------------------------
# 9. Coverage by positive location
# -------------------------------------------------------------------------

coverage = (
    regular.groupby(["location_id", "location"], dropna=False)
    .agg(
        n_rows=("_parsed_date", "size"),
        n_months=("_parsed_date", "nunique"),
        min_date=("_parsed_date", "min"),
        max_date=("_parsed_date", "max")
    )
    .reset_index()
    .sort_values("location_id")
)

print("\n" + "=" * 110)
print("COVERAGE BY POSITIVE LOCATION")
print("=" * 110)

print(coverage.to_string(index=False))

# -------------------------------------------------------------------------
# 10. Final structural summary
# -------------------------------------------------------------------------

print("\n" + "=" * 110)
print("STEP 22.3 STRUCTURAL SUMMARY")
print("=" * 110)

print(f"""
Raw rows                         : {len(df):,}
Unique months                    : {parsed_date.nunique():,}
Date range                       : {parsed_date.min()} to {parsed_date.max()}
Positive location IDs            : {regular['location_id'].nunique():,}
Positive location names          : {regular['location'].nunique(dropna=True):,}
Rows with ID -99                 : {(df['location_id'] == -99).sum():,}
Rows with ID -1                  : {(df['location_id'] == -1).sum():,}
Missing location names           : {df['location'].isna().sum():,}
Exact duplicate raw rows         : {df.duplicated().sum():,}
Duplicate month-location ID rows : {n_duplicate_rows:,}
""")

print("STEP 22.3 STATUS: AUDIT COMPLETE — INTERPRETATION REQUIRED")
print("=" * 110)

In [ ]:
# =============================================================================
# STEP 22.4 — MBIE RENTAL BOND ↔ TA2025 GEOGRAPHY CROSSWALK AUDIT
#
# Purpose:
#   1. Recover the frozen TA2025 geography used by the project.
#   2. Compare MBIE Rental Bond named TAs against TA2025.
#   3. Identify exactly which TA2025 geography is absent from Rental Bond data.
#   4. Test deterministic name matching, including macron/ASCII differences.
#
# IMPORTANT:
#   - Audit only.
#   - No rental features.
#   - No imputation.
#   - No nearest-TA reassignment.
#   - No MCERT processing.
# =============================================================================

from pathlib import Path
import pandas as pd
import unicodedata
import re

print("=" * 110)
print("STEP 22.4 — MBIE RENTAL BOND ↔ TA2025 GEOGRAPHY CROSSWALK AUDIT")
print("=" * 110)

PROJECT_ROOT = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")

GEO_DIR = PROJECT_ROOT / "02_intermediate" / "geography"

# -------------------------------------------------------------------------
# 1. Find frozen listing→TA geography files
# -------------------------------------------------------------------------

geo_files = sorted(GEO_DIR.glob("listing_ta_*.parquet"))

print("\nFROZEN GEOGRAPHY FILES")
print("-" * 110)

print(f"Files found: {len(geo_files)}")

for p in geo_files:
    print(f"  - {p.name}")

if len(geo_files) == 0:
    raise FileNotFoundError(
        "No listing_ta_*.parquet files found in:\n"
        f"{GEO_DIR}"
    )

# -------------------------------------------------------------------------
# 2. Recover canonical TA2025 geography
# -------------------------------------------------------------------------

geo_parts = []

for p in geo_files:

    tmp = pd.read_parquet(
        p,
        columns=["ta_code", "ta_name", "ta_name_ascii"]
    )

    geo_parts.append(tmp)

ta2025 = (
    pd.concat(geo_parts, ignore_index=True)
      .drop_duplicates()
      .sort_values("ta_code")
      .reset_index(drop=True)
)

print("\n" + "=" * 110)
print("CANONICAL TA2025 GEOGRAPHY")
print("=" * 110)

print(f"\nUnique TA2025 records: {len(ta2025):,}")
print(f"Unique TA codes      : {ta2025['ta_code'].nunique():,}")

print("\nTA2025 geography:")
print(ta2025.to_string(index=False))

# -------------------------------------------------------------------------
# 3. Separate actual TAs and TA999
# -------------------------------------------------------------------------

ta2025["ta_code_str"] = (
    ta2025["ta_code"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(3)
)

actual_tas = (
    ta2025.loc[ta2025["ta_code_str"] != "999"]
    .copy()
    .reset_index(drop=True)
)

ta999 = (
    ta2025.loc[ta2025["ta_code_str"] == "999"]
    .copy()
)

print("\n" + "=" * 110)
print("TA2025 CATEGORY COUNTS")
print("=" * 110)

print(f"\nActual TA2025 TAs : {len(actual_tas):,}")
print(f"TA999 categories  : {len(ta999):,}")

# -------------------------------------------------------------------------
# 4. Rental Bond named positive locations
# -------------------------------------------------------------------------

rental_locations = (
    df.loc[
        (df["location_id"] > 0) &
        (df["location"].notna()),
        ["location_id", "location"]
    ]
    .drop_duplicates()
    .sort_values("location_id")
    .reset_index(drop=True)
)

print("\n" + "=" * 110)
print("MBIE RENTAL BOND NAMED LOCATIONS")
print("=" * 110)

print(f"\nRental positive locations: {len(rental_locations):,}")
print(rental_locations.to_string(index=False))

# -------------------------------------------------------------------------
# 5. Deterministic ASCII-normalisation function
#
# Used only for geography-name comparison.
# -------------------------------------------------------------------------

def normalize_name(x):

    if pd.isna(x):
        return None

    x = str(x).strip()

    # Remove macrons/diacritics
    x = unicodedata.normalize("NFKD", x)
    x = "".join(
        ch for ch in x
        if not unicodedata.combining(ch)
    )

    # Standardise case
    x = x.lower()

    # Standardise punctuation / spacing
    x = x.replace("&", "and")
    x = re.sub(r"[^a-z0-9]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()

    return x

rental_locations["name_key"] = (
    rental_locations["location"].map(normalize_name)
)

actual_tas["official_key"] = (
    actual_tas["ta_name"].map(normalize_name)
)

actual_tas["ascii_key"] = (
    actual_tas["ta_name_ascii"].map(normalize_name)
)

# Prefer project's existing ASCII name as deterministic matching key
actual_tas["match_key"] = actual_tas["ascii_key"]

# -------------------------------------------------------------------------
# 6. Check uniqueness of normalized keys
# -------------------------------------------------------------------------

print("\n" + "=" * 110)
print("NORMALIZED KEY UNIQUENESS")
print("=" * 110)

print(
    f"\nRental duplicated normalized keys : "
    f"{rental_locations['name_key'].duplicated().sum():,}"
)

print(
    f"TA2025 duplicated normalized keys : "
    f"{actual_tas['match_key'].duplicated().sum():,}"
)

# -------------------------------------------------------------------------
# 7. Deterministic name crosswalk
# -------------------------------------------------------------------------

crosswalk = rental_locations.merge(
    actual_tas[
        [
            "ta_code_str",
            "ta_name",
            "ta_name_ascii",
            "match_key"
        ]
    ],
    left_on="name_key",
    right_on="match_key",
    how="left",
    validate="one_to_one"
)

print("\n" + "=" * 110)
print("RENTAL → TA2025 CROSSWALK RESULT")
print("=" * 110)

print(
    crosswalk[
        [
            "location_id",
            "location",
            "ta_code_str",
            "ta_name",
            "ta_name_ascii"
        ]
    ].to_string(index=False)
)

# -------------------------------------------------------------------------
# 8. Rental locations that did NOT match TA2025
# -------------------------------------------------------------------------

unmatched_rental = crosswalk.loc[
    crosswalk["ta_code_str"].isna()
].copy()

print("\n" + "=" * 110)
print("UNMATCHED RENTAL LOCATIONS")
print("=" * 110)

print(f"\nCount: {len(unmatched_rental):,}")

if len(unmatched_rental):
    print(
        unmatched_rental[
            ["location_id", "location", "name_key"]
        ].to_string(index=False)
    )
else:
    print("None.")

# -------------------------------------------------------------------------
# 9. TA2025 actual TAs absent from Rental Bond data
# -------------------------------------------------------------------------

matched_codes = set(
    crosswalk["ta_code_str"].dropna().astype(str)
)

missing_ta2025 = actual_tas.loc[
    ~actual_tas["ta_code_str"].isin(matched_codes)
].copy()

print("\n" + "=" * 110)
print("TA2025 ACTUAL TAs ABSENT FROM RENTAL BOND DATA")
print("=" * 110)

print(f"\nCount: {len(missing_ta2025):,}")

if len(missing_ta2025):
    print(
        missing_ta2025[
            ["ta_code_str", "ta_name", "ta_name_ascii"]
        ].to_string(index=False)
    )
else:
    print("None.")

# -------------------------------------------------------------------------
# 10. Check whether TA999 has a direct Rental Bond equivalent
# -------------------------------------------------------------------------

print("\n" + "=" * 110)
print("TA999 CHECK")
print("=" * 110)

if len(ta999):

    print(
        ta999[
            ["ta_code_str", "ta_name", "ta_name_ascii"]
        ].to_string(index=False)
    )

else:
    print("TA999 not found in frozen project geography.")

print("\nRental special IDs:")
print(
    df.loc[
        df["location_id"].isin([-99, -1]),
        ["location_id", "location"]
    ]
    .drop_duplicates()
    .to_string(index=False)
)

# -------------------------------------------------------------------------
# 11. Final summary
# -------------------------------------------------------------------------

print("\n" + "=" * 110)
print("STEP 22.4 SUMMARY")
print("=" * 110)

print(f"""
TA2025 actual TAs                         : {len(actual_tas):,}
Rental Bond positive named locations     : {len(rental_locations):,}
Matched Rental locations                 : {crosswalk['ta_code_str'].notna().sum():,}
Unmatched Rental locations               : {len(unmatched_rental):,}
TA2025 actual TAs absent from Rental     : {len(missing_ta2025):,}
Rental special ID -99 rows               : {(df['location_id'] == -99).sum():,}
Rental special ID -1 rows                : {(df['location_id'] == -1).sum():,}
""")

if (
    len(rental_locations) == 66
    and len(unmatched_rental) == 0
    and len(missing_ta2025) == 1
):
    print("STEP 22.4 STATUS: STRUCTURAL CROSSWALK PASS")
    print("Exactly one TA2025 actual TA is absent from Rental Bond data.")
else:
    print("STEP 22.4 STATUS: REVIEW REQUIRED")

print("=" * 110)

In [ ]:
# =============================================================================
# STEP 22.5 — MBIE RENTAL BOND RECENT-PERIOD COMPLETENESS + VALUE AUDIT
#
# Audit window:
#   2024-01 through 2026-07
#
# Purpose:
#   1. Test whether all 66 Rental Bond TAs are represented in recent months.
#   2. Identify missing TA-month combinations.
#   3. Audit zeros and suspicious values.
#   4. Audit distributions of rent and bond measures.
#   5. Inspect Chatham/TA999 implications WITHOUT imputing anything.
#
# IMPORTANT:
#   - Audit only.
#   - No feature engineering.
#   - No temporal lag rule yet.
#   - No imputation.
#   - No zero-filling missing TA-months.
# =============================================================================

import pandas as pd
import numpy as np

print("=" * 110)
print("STEP 22.5 — MBIE RENTAL BOND RECENT-PERIOD COMPLETENESS + VALUE AUDIT")
print("=" * 110)

# -----------------------------------------------------------------------------
# 1. Create temporary parsed copy of positive named TAs only
# -----------------------------------------------------------------------------

recent = df.loc[
    (df["location_id"] > 0) &
    (df["location"].notna())
].copy()

recent["_date"] = pd.to_datetime(
    recent["TimeFrame"],
    format="%d/%m/%Y",
    errors="coerce"
)

# Restrict audit to 2024-01 through 2026-07
START_DATE = pd.Timestamp("2024-01-01")
END_DATE   = pd.Timestamp("2026-07-01")

recent = recent.loc[
    recent["_date"].between(START_DATE, END_DATE)
].copy()

print("\nRECENT AUDIT WINDOW")
print("-" * 110)

print(f"Start date              : {START_DATE.date()}")
print(f"End date                : {END_DATE.date()}")
print(f"Expected months         : {len(pd.date_range(START_DATE, END_DATE, freq='MS'))}")
print(f"Observed rows           : {len(recent):,}")
print(f"Observed months         : {recent['_date'].nunique():,}")
print(f"Observed location IDs   : {recent['location_id'].nunique():,}")
print(f"Observed location names : {recent['location'].nunique():,}")

# -----------------------------------------------------------------------------
# 2. Month × TA completeness
# -----------------------------------------------------------------------------

expected_months = pd.date_range(
    START_DATE,
    END_DATE,
    freq="MS"
)

expected_ids = sorted(
    rental_locations["location_id"].unique()
)

expected_grid = pd.MultiIndex.from_product(
    [expected_months, expected_ids],
    names=["_date", "location_id"]
).to_frame(index=False)

observed_keys = (
    recent[["_date", "location_id"]]
    .drop_duplicates()
)

grid_check = expected_grid.merge(
    observed_keys.assign(observed=1),
    on=["_date", "location_id"],
    how="left"
)

grid_check["observed"] = (
    grid_check["observed"]
    .fillna(0)
    .astype(int)
)

missing_grid = grid_check.loc[
    grid_check["observed"] == 0
].copy()

print("\n" + "=" * 110)
print("RECENT MONTH × TA COMPLETENESS")
print("=" * 110)

print(f"\nExpected TA-month combinations : {len(expected_grid):,}")
print(f"Observed combinations          : {len(observed_keys):,}")
print(f"Missing combinations           : {len(missing_grid):,}")

# -----------------------------------------------------------------------------
# 3. Number of TAs represented each month
# -----------------------------------------------------------------------------

tas_per_month = (
    recent.groupby("_date")["location_id"]
    .nunique()
    .reindex(expected_months, fill_value=0)
)

print("\nTAs represented per month:")
print(
    tas_per_month
    .rename("n_tas")
    .to_string()
)

print("\nSummary:")
print(f"Minimum TAs/month : {tas_per_month.min():,}")
print(f"Maximum TAs/month : {tas_per_month.max():,}")

# -----------------------------------------------------------------------------
# 4. Missing combinations, if any
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("MISSING RECENT TA-MONTH COMBINATIONS")
print("=" * 110)

if len(missing_grid) == 0:
    print("\nNone.")
else:

    missing_with_name = missing_grid.merge(
        rental_locations[
            ["location_id", "location"]
        ],
        on="location_id",
        how="left",
        validate="many_to_one"
    )

    print(f"\nMissing combinations: {len(missing_with_name):,}")

    print(
        missing_with_name[
            ["_date", "location_id", "location"]
        ]
        .sort_values(["_date", "location_id"])
        .to_string(index=False)
    )

# -----------------------------------------------------------------------------
# 5. Duplicate recent keys
# -----------------------------------------------------------------------------

dup_recent = recent.duplicated(
    subset=["_date", "location_id"],
    keep=False
)

print("\n" + "=" * 110)
print("RECENT KEY UNIQUENESS")
print("=" * 110)

print(
    f"\nRows involved in duplicate month × TA keys: "
    f"{dup_recent.sum():,}"
)

# -----------------------------------------------------------------------------
# 6. Value fields
# -----------------------------------------------------------------------------

value_cols = [
    "LodgedBonds",
    "ActiveBonds",
    "ClosedBonds",
    "MedianRent",
    "GeometricMeanRent",
    "UpperQuartileRent",
    "LowerQuartileRent",
    "LogStdDevWeeklyRent"
]

print("\n" + "=" * 110)
print("RECENT MISSING / ZERO / NEGATIVE VALUE AUDIT")
print("=" * 110)

audit_rows = []

for c in value_cols:

    s = recent[c]

    audit_rows.append({
        "variable": c,
        "n": len(s),
        "missing_n": int(s.isna().sum()),
        "zero_n": int((s == 0).sum()),
        "negative_n": int((s < 0).sum()),
        "min": s.min(),
        "max": s.max()
    })

value_audit = pd.DataFrame(audit_rows)

print(value_audit.to_string(index=False))

# -----------------------------------------------------------------------------
# 7. Distribution audit
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("RECENT DISTRIBUTION AUDIT")
print("=" * 110)

distribution_rows = []

for c in value_cols:

    s = recent[c].dropna()

    distribution_rows.append({
        "variable": c,
        "count": len(s),
        "mean": s.mean(),
        "std": s.std(),
        "min": s.min(),
        "p01": s.quantile(0.01),
        "p05": s.quantile(0.05),
        "p25": s.quantile(0.25),
        "median": s.quantile(0.50),
        "p75": s.quantile(0.75),
        "p95": s.quantile(0.95),
        "p99": s.quantile(0.99),
        "max": s.max()
    })

distribution = pd.DataFrame(distribution_rows)

print(
    distribution.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

# -----------------------------------------------------------------------------
# 8. Rent-order consistency
#
# Expected:
# Lower quartile <= Median <= Upper quartile
# -----------------------------------------------------------------------------

rent_order_bad = recent.loc[
    ~(
        (recent["LowerQuartileRent"] <= recent["MedianRent"]) &
        (recent["MedianRent"] <= recent["UpperQuartileRent"])
    )
].copy()

print("\n" + "=" * 110)
print("RENT QUARTILE ORDER CONSISTENCY")
print("=" * 110)

print(
    f"\nRows violating Lower <= Median <= Upper: "
    f"{len(rent_order_bad):,}"
)

if len(rent_order_bad) > 0:
    print(
        rent_order_bad[
            [
                "_date",
                "location_id",
                "location",
                "LowerQuartileRent",
                "MedianRent",
                "UpperQuartileRent"
            ]
        ]
        .head(30)
        .to_string(index=False)
    )

# -----------------------------------------------------------------------------
# 9. Coverage by TA in recent audit window
# -----------------------------------------------------------------------------

recent_coverage = (
    recent.groupby(
        ["location_id", "location"],
        as_index=False
    )
    .agg(
        n_months=("_date", "nunique"),
        min_date=("_date", "min"),
        max_date=("_date", "max"),
        median_rent_min=("MedianRent", "min"),
        median_rent_max=("MedianRent", "max")
    )
    .sort_values("location_id")
)

print("\n" + "=" * 110)
print("RECENT COVERAGE BY RENTAL TA")
print("=" * 110)

print(recent_coverage.to_string(index=False))

# -----------------------------------------------------------------------------
# 10. Special records during recent period
# -----------------------------------------------------------------------------

special_recent = df.loc[
    df["location_id"].isin([-99, -1])
].copy()

special_recent["_date"] = pd.to_datetime(
    special_recent["TimeFrame"],
    format="%d/%m/%Y",
    errors="coerce"
)

special_recent = special_recent.loc[
    special_recent["_date"].between(
        START_DATE,
        END_DATE
    )
]

print("\n" + "=" * 110)
print("SPECIAL RECORDS — RECENT WINDOW")
print("=" * 110)

print(
    special_recent.groupby(
        ["location_id", "location"],
        dropna=False
    )
    .agg(
        rows=("_date", "size"),
        months=("_date", "nunique"),
        min_date=("_date", "min"),
        max_date=("_date", "max")
    )
    .reset_index()
    .to_string(index=False)
)

# -----------------------------------------------------------------------------
# 11. Final summary
# -----------------------------------------------------------------------------

expected_n = len(expected_months) * len(expected_ids)

print("\n" + "=" * 110)
print("STEP 22.5 SUMMARY")
print("=" * 110)

print(f"""
Audit period                         : 2024-01 through 2026-07
Expected months                      : {len(expected_months):,}
Expected Rental TAs                  : {len(expected_ids):,}
Expected TA-month combinations       : {expected_n:,}
Observed TA-month combinations       : {len(observed_keys):,}
Missing TA-month combinations        : {len(missing_grid):,}
Duplicate recent month-TA rows       : {dup_recent.sum():,}
Minimum TAs represented per month    : {tas_per_month.min():,}
Maximum TAs represented per month    : {tas_per_month.max():,}
""")

if (
    len(missing_grid) == 0
    and dup_recent.sum() == 0
    and tas_per_month.min() == len(expected_ids)
):
    print("STEP 22.5 STRUCTURAL COVERAGE STATUS: PASS")
    print("All 66 Rental Bond TAs are represented in every month of the recent audit window.")
else:
    print("STEP 22.5 STRUCTURAL COVERAGE STATUS: REVIEW REQUIRED")

print("=" * 110)

In [ ]:
# =============================================================================
# STEP 22.7 — RENTAL BOND T1–T10 POINT-IN-TIME INFORMATION-SET AUDIT
#
# Purpose:
#   1. Apply the frozen conservative Rental Bond availability rule.
#   2. Extract exactly one eligible rental month for each T1–T10 anchor.
#   3. Audit TA coverage and suppressed/missing TA-months.
#   4. Audit key candidate variables.
#
# IMPORTANT:
#   - QC only.
#   - No feature engineering.
#   - No interpolation.
#   - No zero filling.
#   - No rolling averages.
#   - No YoY calculations.
#   - No feature freezing yet.
# =============================================================================

import pandas as pd
import numpy as np

print("=" * 110)
print("STEP 22.7 — RENTAL BOND T1–T10 POINT-IN-TIME INFORMATION-SET AUDIT")
print("=" * 110)

# -----------------------------------------------------------------------------
# 1. Frozen prediction origins and conservative eligible rental months
# -----------------------------------------------------------------------------

alignment = pd.DataFrame({
    "transition": [
        "T1", "T2", "T3", "T4", "T5",
        "T6", "T7", "T8", "T9", "T10"
    ],

    "anchor_date": pd.to_datetime([
        "2025-10-05",
        "2025-11-07",
        "2025-12-11",
        "2026-01-16",
        "2026-02-13",
        "2026-03-17",
        "2026-04-16",
        "2026-05-23",
        "2026-06-19",
        "2026-07-12"
    ]),

    "eligible_rental_month": pd.to_datetime([
        "2025-08-01",
        "2025-09-01",
        "2025-10-01",
        "2025-11-01",
        "2025-12-01",
        "2026-01-01",
        "2026-02-01",
        "2026-03-01",
        "2026-04-01",
        "2026-05-01"
    ])
})

alignment["information_lag_days"] = (
    alignment["anchor_date"]
    - alignment["eligible_rental_month"]
).dt.days

print("\nFROZEN TEMPORAL ALIGNMENT")
print("-" * 110)

print(alignment.to_string(index=False))

# -----------------------------------------------------------------------------
# 2. Prepare positive named Rental Bond TAs
# -----------------------------------------------------------------------------

rental = df.loc[
    (df["location_id"] > 0) &
    (df["location"].notna())
].copy()

rental["rental_month"] = pd.to_datetime(
    rental["TimeFrame"],
    format="%d/%m/%Y",
    errors="coerce"
)

print("\n" + "=" * 110)
print("SOURCE GEOGRAPHY")
print("=" * 110)

print(f"\nPositive named Rental Bond TAs: {rental['location_id'].nunique():,}")

# -----------------------------------------------------------------------------
# 3. Extract only the 10 eligible information months
# -----------------------------------------------------------------------------

eligible_months = set(alignment["eligible_rental_month"])

audit = rental.loc[
    rental["rental_month"].isin(eligible_months)
].copy()

audit = audit.merge(
    alignment[
        [
            "transition",
            "anchor_date",
            "eligible_rental_month",
            "information_lag_days"
        ]
    ],
    left_on="rental_month",
    right_on="eligible_rental_month",
    how="left",
    validate="many_to_one"
)

print("\n" + "=" * 110)
print("EXTRACTED T1–T10 INFORMATION SET")
print("=" * 110)

print(f"\nRows extracted       : {len(audit):,}")
print(f"Transitions          : {audit['transition'].nunique():,}")
print(f"Rental months        : {audit['rental_month'].nunique():,}")
print(f"Unique Rental TAs    : {audit['location_id'].nunique():,}")

# -----------------------------------------------------------------------------
# 4. Expected transition × TA grid
# -----------------------------------------------------------------------------

expected_ids = (
    rental[["location_id", "location"]]
    .drop_duplicates()
    .sort_values("location_id")
)

expected_grid = (
    alignment[["transition", "anchor_date", "eligible_rental_month"]]
    .merge(expected_ids, how="cross")
)

observed_keys = (
    audit[
        [
            "transition",
            "location_id"
        ]
    ]
    .drop_duplicates()
)

coverage = expected_grid.merge(
    observed_keys.assign(observed=1),
    on=["transition", "location_id"],
    how="left"
)

coverage["observed"] = (
    coverage["observed"]
    .fillna(0)
    .astype(int)
)

missing = coverage.loc[
    coverage["observed"] == 0
].copy()

print("\n" + "=" * 110)
print("TRANSITION × TA COVERAGE")
print("=" * 110)

print(f"\nExpected combinations : {len(expected_grid):,}")
print(f"Observed combinations : {len(observed_keys):,}")
print(f"Missing combinations  : {len(missing):,}")

# -----------------------------------------------------------------------------
# 5. Number of TAs available at each transition
# -----------------------------------------------------------------------------

transition_coverage = (
    audit.groupby(
        [
            "transition",
            "anchor_date",
            "eligible_rental_month"
        ],
        as_index=False
    )
    .agg(
        n_tas=("location_id", "nunique")
    )
)

# Preserve T1 -> T10 ordering
transition_order = [
    "T1", "T2", "T3", "T4", "T5",
    "T6", "T7", "T8", "T9", "T10"
]

transition_coverage["transition"] = pd.Categorical(
    transition_coverage["transition"],
    categories=transition_order,
    ordered=True
)

transition_coverage = (
    transition_coverage
    .sort_values("transition")
)

print("\nCOVERAGE BY TRANSITION")
print("-" * 110)

print(transition_coverage.to_string(index=False))

# -----------------------------------------------------------------------------
# 6. Exact missing TA-transition combinations
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("MISSING / SUPPRESSED TA-TRANSITION COMBINATIONS")
print("=" * 110)

if len(missing) == 0:

    print("\nNone.")

else:

    missing = (
        missing
        .sort_values(
            ["eligible_rental_month", "location_id"]
        )
    )

    print(f"\nCount: {len(missing):,}\n")

    print(
        missing[
            [
                "transition",
                "anchor_date",
                "eligible_rental_month",
                "location_id",
                "location"
            ]
        ].to_string(index=False)
    )

# -----------------------------------------------------------------------------
# 7. Duplicate key audit
# -----------------------------------------------------------------------------

duplicate_keys = audit.duplicated(
    subset=["transition", "location_id"],
    keep=False
)

print("\n" + "=" * 110)
print("KEY UNIQUENESS")
print("=" * 110)

print(
    f"\nRows involved in duplicate transition × TA keys: "
    f"{duplicate_keys.sum():,}"
)

# -----------------------------------------------------------------------------
# 8. Candidate-variable audit
# -----------------------------------------------------------------------------

candidate_cols = [
    "GeometricMeanRent",
    "MedianRent",
    "LodgedBonds",
    "ActiveBonds",
    "ClosedBonds",
    "UpperQuartileRent",
    "LowerQuartileRent",
    "LogStdDevWeeklyRent"
]

candidate_audit = []

for c in candidate_cols:

    s = audit[c]

    candidate_audit.append({
        "variable": c,
        "n": len(s),
        "missing_n": int(s.isna().sum()),
        "zero_n": int((s == 0).sum()),
        "negative_n": int((s < 0).sum()),
        "min": s.min(),
        "median": s.median(),
        "mean": s.mean(),
        "max": s.max()
    })

candidate_audit = pd.DataFrame(candidate_audit)

print("\n" + "=" * 110)
print("CANDIDATE VARIABLE AUDIT — T1–T10 INFORMATION SETS")
print("=" * 110)

print(
    candidate_audit.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

# -----------------------------------------------------------------------------
# 9. GeometricMeanRent coverage by transition
# -----------------------------------------------------------------------------

rent_transition = (
    audit.groupby("transition", observed=False)
    .agg(
        n_records=("location_id", "size"),
        n_tas=("location_id", "nunique"),
        geometric_mean_rent_min=("GeometricMeanRent", "min"),
        geometric_mean_rent_median=("GeometricMeanRent", "median"),
        geometric_mean_rent_max=("GeometricMeanRent", "max"),
        median_rent_median=("MedianRent", "median"),
        lodged_bonds_median=("LodgedBonds", "median"),
        active_bonds_median=("ActiveBonds", "median")
    )
    .reset_index()
)

rent_transition["transition"] = pd.Categorical(
    rent_transition["transition"],
    categories=transition_order,
    ordered=True
)

rent_transition = (
    rent_transition
    .sort_values("transition")
)

print("\n" + "=" * 110)
print("VARIABLE SUMMARY BY TRANSITION")
print("=" * 110)

print(
    rent_transition.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

# -----------------------------------------------------------------------------
# 10. Quartile-order consistency
# -----------------------------------------------------------------------------

bad_quartiles = audit.loc[
    ~(
        (audit["LowerQuartileRent"] <= audit["MedianRent"]) &
        (audit["MedianRent"] <= audit["UpperQuartileRent"])
    )
]

print("\n" + "=" * 110)
print("RENT QUARTILE CONSISTENCY")
print("=" * 110)

print(
    f"\nRows violating Lower <= Median <= Upper: "
    f"{len(bad_quartiles):,}"
)

# -----------------------------------------------------------------------------
# 11. Check special geography handling
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("PROJECT GEOGRAPHY IMPLICATION")
print("=" * 110)

print("""
Rental Bond named TA coverage universe = 66 TAs.

TA067 — Chatham Islands Territory:
    Rental Bond context unavailable -> retain as NaN later.

TA999 — Area Outside Territorial Authority:
    Rental Bond context unavailable -> retain as NaN later.

No nearest-TA reassignment.
No zero filling.
""")

# -----------------------------------------------------------------------------
# 12. Final summary
# -----------------------------------------------------------------------------

print("=" * 110)
print("STEP 22.7 SUMMARY")
print("=" * 110)

print(f"""
Transitions                         : {audit['transition'].nunique():,}
Expected transition × TA records    : {len(expected_grid):,}
Observed transition × TA records    : {len(observed_keys):,}
Missing transition × TA records     : {len(missing):,}
Duplicate transition × TA rows      : {duplicate_keys.sum():,}
Minimum TAs in one transition       : {transition_coverage['n_tas'].min():,}
Maximum TAs in one transition       : {transition_coverage['n_tas'].max():,}
""")

if (
    audit["transition"].nunique() == 10
    and duplicate_keys.sum() == 0
):

    print("STEP 22.7 STATUS: POINT-IN-TIME INFORMATION SET CONSTRUCTED")
    print(
        "Coverage gaps, if present, must remain missing/unavailable "
        "until a later modelling-stage missing-data policy is frozen."
    )

else:

    print("STEP 22.7 STATUS: REVIEW REQUIRED")

print("=" * 110)

In [ ]:
# =============================================================================
# STEP 22.8A — RENTAL-MARKET CANDIDATE FEATURE AUDIT
#
# Purpose:
#   Evaluate point-in-time-safe candidate transformations using
#   GeometricMeanRent as the primary rent-series candidate.
#
# Candidate concepts:
#   1. Current eligible rent level
#   2. 3-month rent change
#   3. 12-month / YoY rent change
#
# IMPORTANT:
#   - No imputation.
#   - No interpolation.
#   - No zero filling.
#   - No feature freezing yet.
#   - Missing historical observations remain NaN.
# =============================================================================

import pandas as pd
import numpy as np

print("=" * 110)
print("STEP 22.8A — RENTAL-MARKET CANDIDATE FEATURE AUDIT")
print("=" * 110)

# -------------------------------------------------------------------------
# 1. Prepare complete historical source for the 66 named Rental Bond TAs
# -------------------------------------------------------------------------

hist = df.loc[
    (df["location_id"] > 0) &
    (df["location"].notna())
].copy()

hist["rental_month"] = pd.to_datetime(
    hist["TimeFrame"],
    format="%d/%m/%Y",
    errors="coerce"
)

hist = hist.sort_values(
    ["location_id", "rental_month"]
).reset_index(drop=True)

# Keep fields needed for this audit
hist = hist[
    [
        "rental_month",
        "location_id",
        "location",
        "GeometricMeanRent",
        "MedianRent",
        "LodgedBonds",
        "ActiveBonds"
    ]
].copy()

# -------------------------------------------------------------------------
# 2. Self-join exact historical months
#
# IMPORTANT:
# These are calendar-month joins, NOT row shifts.
# If an earlier TA-month is absent/suppressed, result remains NaN.
# -------------------------------------------------------------------------

base = hist.copy()

lag3 = hist[
    ["rental_month", "location_id", "GeometricMeanRent"]
].copy()

lag3["rental_month"] = (
    lag3["rental_month"] + pd.DateOffset(months=3)
)

lag3 = lag3.rename(
    columns={
        "GeometricMeanRent": "GeometricMeanRent_3m_ago"
    }
)

lag12 = hist[
    ["rental_month", "location_id", "GeometricMeanRent"]
].copy()

lag12["rental_month"] = (
    lag12["rental_month"] + pd.DateOffset(months=12)
)

lag12 = lag12.rename(
    columns={
        "GeometricMeanRent": "GeometricMeanRent_12m_ago"
    }
)

base = base.merge(
    lag3,
    on=["rental_month", "location_id"],
    how="left",
    validate="one_to_one"
)

base = base.merge(
    lag12,
    on=["rental_month", "location_id"],
    how="left",
    validate="one_to_one"
)

# -------------------------------------------------------------------------
# 3. Construct candidate transformations
#
# These are temporary QC candidates only.
# -------------------------------------------------------------------------

base["rent_level_log"] = np.log(
    base["GeometricMeanRent"]
)

base["rent_change_3m_log"] = (
    np.log(base["GeometricMeanRent"])
    - np.log(base["GeometricMeanRent_3m_ago"])
)

base["rent_change_12m_log"] = (
    np.log(base["GeometricMeanRent"])
    - np.log(base["GeometricMeanRent_12m_ago"])
)

# Percentage versions for interpretation/QC only
base["rent_change_3m_pct"] = (
    (
        base["GeometricMeanRent"]
        / base["GeometricMeanRent_3m_ago"]
    ) - 1
) * 100

base["rent_change_12m_pct"] = (
    (
        base["GeometricMeanRent"]
        / base["GeometricMeanRent_12m_ago"]
    ) - 1
) * 100

# -------------------------------------------------------------------------
# 4. Restrict to the 10 frozen eligible information months
# -------------------------------------------------------------------------

candidate = alignment[
    [
        "transition",
        "anchor_date",
        "eligible_rental_month",
        "information_lag_days"
    ]
].merge(
    base,
    left_on="eligible_rental_month",
    right_on="rental_month",
    how="left"
)

candidate["transition"] = pd.Categorical(
    candidate["transition"],
    categories=[
        "T1", "T2", "T3", "T4", "T5",
        "T6", "T7", "T8", "T9", "T10"
    ],
    ordered=True
)

candidate = candidate.sort_values(
    ["transition", "location_id"]
).reset_index(drop=True)

print("\nINFORMATION SET")
print("-" * 110)

print(f"Rows                         : {len(candidate):,}")
print(f"Transitions                  : {candidate['transition'].nunique():,}")
print(f"Unique TAs represented       : {candidate['location_id'].nunique():,}")

# -------------------------------------------------------------------------
# 5. Missingness audit
# -------------------------------------------------------------------------

audit_cols = [
    "GeometricMeanRent",
    "GeometricMeanRent_3m_ago",
    "GeometricMeanRent_12m_ago",
    "rent_level_log",
    "rent_change_3m_log",
    "rent_change_12m_log",
    "rent_change_3m_pct",
    "rent_change_12m_pct"
]

missing_rows = []

for c in audit_cols:

    missing_rows.append({
        "variable": c,
        "n": len(candidate),
        "available_n": candidate[c].notna().sum(),
        "missing_n": candidate[c].isna().sum(),
        "missing_pct": candidate[c].isna().mean() * 100
    })

missing_audit = pd.DataFrame(missing_rows)

print("\n" + "=" * 110)
print("CANDIDATE MISSINGNESS")
print("=" * 110)

print(
    missing_audit.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

# -------------------------------------------------------------------------
# 6. Missingness by transition
# -------------------------------------------------------------------------

transition_missing = (
    candidate.groupby(
        "transition",
        observed=False
    )
    .agg(
        n_rows=("location_id", "size"),

        current_rent_available=(
            "GeometricMeanRent",
            lambda x: x.notna().sum()
        ),

        lag3_available=(
            "GeometricMeanRent_3m_ago",
            lambda x: x.notna().sum()
        ),

        lag12_available=(
            "GeometricMeanRent_12m_ago",
            lambda x: x.notna().sum()
        ),

        change3_available=(
            "rent_change_3m_log",
            lambda x: x.notna().sum()
        ),

        change12_available=(
            "rent_change_12m_log",
            lambda x: x.notna().sum()
        )
    )
    .reset_index()
)

print("\n" + "=" * 110)
print("AVAILABILITY BY TRANSITION")
print("=" * 110)

print(transition_missing.to_string(index=False))

# -------------------------------------------------------------------------
# 7. Distribution audit
# -------------------------------------------------------------------------

feature_cols = [
    "rent_level_log",
    "rent_change_3m_log",
    "rent_change_12m_log",
    "rent_change_3m_pct",
    "rent_change_12m_pct"
]

dist_rows = []

for c in feature_cols:

    s = candidate[c].dropna()

    dist_rows.append({
        "variable": c,
        "n": len(s),
        "mean": s.mean(),
        "std": s.std(),
        "min": s.min(),
        "p01": s.quantile(0.01),
        "p05": s.quantile(0.05),
        "p25": s.quantile(0.25),
        "median": s.median(),
        "p75": s.quantile(0.75),
        "p95": s.quantile(0.95),
        "p99": s.quantile(0.99),
        "max": s.max()
    })

dist = pd.DataFrame(dist_rows)

print("\n" + "=" * 110)
print("CANDIDATE DISTRIBUTIONS")
print("=" * 110)

print(
    dist.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# -------------------------------------------------------------------------
# 8. Extreme-change audit
#
# Not automatically errors.
# We inspect them before feature freezing.
# -------------------------------------------------------------------------

print("\n" + "=" * 110)
print("LARGEST ABSOLUTE 3-MONTH RENT CHANGES")
print("=" * 110)

tmp3 = candidate.loc[
    candidate["rent_change_3m_pct"].notna()
].copy()

tmp3["abs_change"] = (
    tmp3["rent_change_3m_pct"].abs()
)

print(
    tmp3.sort_values(
        "abs_change",
        ascending=False
    )[
        [
            "transition",
            "eligible_rental_month",
            "location_id",
            "location",
            "GeometricMeanRent_3m_ago",
            "GeometricMeanRent",
            "rent_change_3m_pct"
        ]
    ]
    .head(20)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

print("\n" + "=" * 110)
print("LARGEST ABSOLUTE 12-MONTH RENT CHANGES")
print("=" * 110)

tmp12 = candidate.loc[
    candidate["rent_change_12m_pct"].notna()
].copy()

tmp12["abs_change"] = (
    tmp12["rent_change_12m_pct"].abs()
)

print(
    tmp12.sort_values(
        "abs_change",
        ascending=False
    )[
        [
            "transition",
            "eligible_rental_month",
            "location_id",
            "location",
            "GeometricMeanRent_12m_ago",
            "GeometricMeanRent",
            "rent_change_12m_pct"
        ]
    ]
    .head(20)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

# -------------------------------------------------------------------------
# 9. Correlation among candidate rent representations
#
# Descriptive QC only — NOT model selection.
# -------------------------------------------------------------------------

corr_cols = [
    "GeometricMeanRent",
    "rent_level_log",
    "rent_change_3m_log",
    "rent_change_12m_log"
]

corr = candidate[corr_cols].corr()

print("\n" + "=" * 110)
print("CANDIDATE CORRELATION — DESCRIPTIVE ONLY")
print("=" * 110)

print(
    corr.to_string(
        float_format=lambda x: f"{x:.3f}"
    )
)

# -------------------------------------------------------------------------
# 10. Final summary
# -------------------------------------------------------------------------

print("\n" + "=" * 110)
print("STEP 22.8A SUMMARY")
print("=" * 110)

print(f"""
Information-set rows                  : {len(candidate):,}
Current GeometricMeanRent available   : {candidate['GeometricMeanRent'].notna().sum():,}
3-month change available              : {candidate['rent_change_3m_log'].notna().sum():,}
12-month change available             : {candidate['rent_change_12m_log'].notna().sum():,}
""")

print("STEP 22.8A STATUS: CANDIDATE TRANSFORMATIONS AUDITED — NOT YET FROZEN")
print("=" * 110)

In [ ]:
# =============================================================================
# STEP 22.8B — RENT CHANGE RELIABILITY / SMALL-MARKET AUDIT
#
# Purpose:
#   Examine whether extreme 12-month rent changes are associated with
#   low Rental Bond activity.
#
# QC ONLY:
#   - No winsorisation
#   - No filtering
#   - No imputation
#   - No feature freezing
# =============================================================================

import pandas as pd
import numpy as np

print("=" * 110)
print("STEP 22.8B — RENT CHANGE RELIABILITY / SMALL-MARKET AUDIT")
print("=" * 110)

# -------------------------------------------------------------------------
# 1. Add prior-year LodgedBonds using exact calendar matching
# -------------------------------------------------------------------------

lag12_bonds = hist[
    [
        "rental_month",
        "location_id",
        "LodgedBonds"
    ]
].copy()

lag12_bonds["rental_month"] = (
    lag12_bonds["rental_month"]
    + pd.DateOffset(months=12)
)

lag12_bonds = lag12_bonds.rename(
    columns={
        "LodgedBonds": "LodgedBonds_12m_ago"
    }
)

qc = candidate.merge(
    lag12_bonds,
    on=["rental_month", "location_id"],
    how="left",
    validate="one_to_one"
)

# -------------------------------------------------------------------------
# 2. Absolute 12-month change
# -------------------------------------------------------------------------

qc["abs_rent_change_12m_pct"] = (
    qc["rent_change_12m_pct"].abs()
)

print("\nROWS AVAILABLE FOR 12M RELIABILITY AUDIT")
print("-" * 110)

valid = qc.loc[
    qc["rent_change_12m_pct"].notna()
].copy()

print(f"Valid 12M changes : {len(valid):,}")
print(
    f"Prior-year bond count available : "
    f"{valid['LodgedBonds_12m_ago'].notna().sum():,}"
)

# -------------------------------------------------------------------------
# 3. Largest 12M changes + bond activity
# -------------------------------------------------------------------------

print("\n" + "=" * 110)
print("TOP 30 ABSOLUTE 12-MONTH RENT CHANGES + BOND ACTIVITY")
print("=" * 110)

cols = [
    "transition",
    "eligible_rental_month",
    "location_id",
    "location",
    "GeometricMeanRent_12m_ago",
    "GeometricMeanRent",
    "rent_change_12m_pct",
    "LodgedBonds_12m_ago",
    "LodgedBonds"
]

print(
    valid.sort_values(
        "abs_rent_change_12m_pct",
        ascending=False
    )[cols]
    .head(30)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

# -------------------------------------------------------------------------
# 4. Change distribution by current bond-volume band
#
# Descriptive only.
# -------------------------------------------------------------------------

valid["bond_volume_band"] = pd.cut(
    valid["LodgedBonds"],
    bins=[
        -np.inf,
        14,
        29,
        59,
        119,
        np.inf
    ],
    labels=[
        "6-14",
        "15-29",
        "30-59",
        "60-119",
        "120+"
    ]
)

band_summary = (
    valid.groupby(
        "bond_volume_band",
        observed=False
    )
    .agg(
        n=("rent_change_12m_pct", "size"),
        median_bonds=("LodgedBonds", "median"),
        median_abs_change=(
            "abs_rent_change_12m_pct",
            "median"
        ),
        p90_abs_change=(
            "abs_rent_change_12m_pct",
            lambda x: x.quantile(0.90)
        ),
        p95_abs_change=(
            "abs_rent_change_12m_pct",
            lambda x: x.quantile(0.95)
        ),
        max_abs_change=(
            "abs_rent_change_12m_pct",
            "max"
        )
    )
    .reset_index()
)

print("\n" + "=" * 110)
print("ABSOLUTE 12M CHANGE BY CURRENT BOND-VOLUME BAND")
print("=" * 110)

print(
    band_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

# -------------------------------------------------------------------------
# 5. Threshold audit
# -------------------------------------------------------------------------

thresholds = [
    10,
    20,
    30,
    50,
    100
]

rows = []

for threshold in thresholds:

    small = valid.loc[
        valid["LodgedBonds"] < threshold
    ]

    large = valid.loc[
        valid["LodgedBonds"] >= threshold
    ]

    rows.append({
        "bond_threshold": threshold,

        "n_below": len(small),
        "median_abs_change_below":
            small["abs_rent_change_12m_pct"].median(),

        "n_at_or_above": len(large),
        "median_abs_change_at_or_above":
            large["abs_rent_change_12m_pct"].median()
    })

threshold_audit = pd.DataFrame(rows)

print("\n" + "=" * 110)
print("BOND-VOLUME THRESHOLD AUDIT")
print("=" * 110)

print(
    threshold_audit.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

# -------------------------------------------------------------------------
# 6. Association between bond volume and instability
# -------------------------------------------------------------------------

pearson = valid[
    [
        "LodgedBonds",
        "abs_rent_change_12m_pct"
    ]
].corr().iloc[0, 1]

spearman = valid[
    [
        "LodgedBonds",
        "abs_rent_change_12m_pct"
    ]
].corr(method="spearman").iloc[0, 1]

print("\n" + "=" * 110)
print("BOND VOLUME ↔ ABSOLUTE RENT-CHANGE ASSOCIATION")
print("=" * 110)

print(f"\nPearson correlation : {pearson:.4f}")
print(f"Spearman correlation: {spearman:.4f}")

# -------------------------------------------------------------------------
# 7. Extreme-change counts
# -------------------------------------------------------------------------

extreme_summary = pd.DataFrame({
    "threshold_abs_pct": [20, 30, 40, 50, 75, 100],
    "n_records": [
        (valid["abs_rent_change_12m_pct"] >= x).sum()
        for x in [20, 30, 40, 50, 75, 100]
    ]
})

extreme_summary["pct_records"] = (
    extreme_summary["n_records"]
    / len(valid)
    * 100
)

print("\n" + "=" * 110)
print("EXTREME 12M CHANGE FREQUENCY")
print("=" * 110)

print(
    extreme_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

# -------------------------------------------------------------------------
# 8. Final summary
# -------------------------------------------------------------------------

print("\n" + "=" * 110)
print("STEP 22.8B SUMMARY")
print("=" * 110)

print(f"""
Valid 12M rent-change records : {len(valid):,}
Median absolute 12M change    : {valid['abs_rent_change_12m_pct'].median():.2f}%
95th percentile abs change    : {valid['abs_rent_change_12m_pct'].quantile(0.95):.2f}%
Maximum absolute change       : {valid['abs_rent_change_12m_pct'].max():.2f}%
Pearson volume/change         : {pearson:.4f}
Spearman volume/change        : {spearman:.4f}
""")

print(
    "STEP 22.8B STATUS: RELIABILITY AUDIT COMPLETE — "
    "FEATURE POLICY NOT YET FROZEN"
)

print("=" * 110)

In [ ]:
# =============================================================================
# STEP 22.9 — BUILD + FREEZE RENTAL-MARKET TA × TRANSITION FEATURE TABLE
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 110)
print("STEP 22.9 — BUILD + FREEZE RENTAL-MARKET TA × TRANSITION FEATURE TABLE")
print("=" * 110)

PROJECT_ROOT = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")

OUT_DIR = PROJECT_ROOT / "02_intermediate" / "property_market"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / "rental_market_features_ta_transition_frozen.parquet"

# -----------------------------------------------------------------------------
# 1. Recover canonical 67 actual TA2025 geographies
# -----------------------------------------------------------------------------

actual_tas_final = (
    actual_tas[
        ["ta_code_str", "ta_name", "ta_name_ascii"]
    ]
    .drop_duplicates()
    .copy()
)

actual_tas_final = actual_tas_final.loc[
    actual_tas_final["ta_code_str"] != "999"
].copy()

actual_tas_final = actual_tas_final.rename(
    columns={"ta_code_str": "ta_code"}
)

print("\nTA2025 GEOGRAPHY")
print("-" * 110)
print(f"Actual TAs: {len(actual_tas_final):,}")

# -----------------------------------------------------------------------------
# 2. Build Rental Bond → TA2025 crosswalk from Step 22.4
# -----------------------------------------------------------------------------

cw = crosswalk[
    ["location_id", "location", "ta_code_str"]
].copy()

cw = cw.rename(
    columns={"ta_code_str": "ta_code"}
)

assert cw["location_id"].nunique() == 66
assert cw["ta_code"].nunique() == 66
assert cw["ta_code"].isna().sum() == 0

# -----------------------------------------------------------------------------
# 3. Take audited candidate values from Step 22.8A
# -----------------------------------------------------------------------------

feature_source = candidate[
    [
        "transition",
        "anchor_date",
        "eligible_rental_month",
        "location_id",
        "location",
        "GeometricMeanRent",
        "rent_level_log",
        "rent_change_12m_log",
        "rent_change_3m_log"
    ]
].copy()

feature_source = feature_source.merge(
    cw[["location_id", "ta_code"]],
    on="location_id",
    how="left",
    validate="many_to_one"
)

assert feature_source["ta_code"].isna().sum() == 0

# Rename to final manuscript/model-safe names
feature_source = feature_source.rename(
    columns={
        "rent_level_log":
            "rental_geomean_rent_log",

        "rent_change_12m_log":
            "rental_geomean_rent_change_12m_log",

        "rent_change_3m_log":
            "rental_geomean_rent_change_3m_log"
    }
)

# -----------------------------------------------------------------------------
# 4. Create complete 10 × 67 TA2025 grid
# -----------------------------------------------------------------------------

transition_base = alignment[
    [
        "transition",
        "anchor_date",
        "eligible_rental_month"
    ]
].copy()

full_grid = transition_base.merge(
    actual_tas_final,
    how="cross"
)

print("\nEXPECTED GRID")
print("-" * 110)

print(f"Transitions : {full_grid['transition'].nunique():,}")
print(f"TAs         : {full_grid['ta_code'].nunique():,}")
print(f"Rows        : {len(full_grid):,}")

# -----------------------------------------------------------------------------
# 5. Merge Rental Bond features onto complete TA2025 grid
# -----------------------------------------------------------------------------

final = full_grid.merge(
    feature_source[
        [
            "transition",
            "ta_code",
            "rental_geomean_rent_log",
            "rental_geomean_rent_change_12m_log",
            "rental_geomean_rent_change_3m_log"
        ]
    ],
    on=["transition", "ta_code"],
    how="left",
    validate="one_to_one"
)

# Technical availability indicators
final["rental_current_available"] = (
    final["rental_geomean_rent_log"].notna()
).astype("int8")

final["rental_change_12m_available"] = (
    final["rental_geomean_rent_change_12m_log"].notna()
).astype("int8")

# -----------------------------------------------------------------------------
# 6. Freeze feature roles
# -----------------------------------------------------------------------------

PRIMARY_FEATURES = [
    "rental_geomean_rent_log",
    "rental_geomean_rent_change_12m_log"
]

ROBUSTNESS_FEATURES = [
    "rental_geomean_rent_change_3m_log"
]

TECHNICAL_FIELDS = [
    "rental_current_available",
    "rental_change_12m_available"
]

print("\n" + "=" * 110)
print("FROZEN FEATURE POLICY")
print("=" * 110)

print("\nPRIMARY:")
for x in PRIMARY_FEATURES:
    print("  -", x)

print("\nROBUSTNESS ONLY:")
for x in ROBUSTNESS_FEATURES:
    print("  -", x)

print("\nTECHNICAL / MISSINGNESS:")
for x in TECHNICAL_FIELDS:
    print("  -", x)

# -----------------------------------------------------------------------------
# 7. Structural QC
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("STRUCTURAL QC")
print("=" * 110)

print(f"\nRows              : {len(final):,}")
print(f"Transitions       : {final['transition'].nunique():,}")
print(f"TA codes          : {final['ta_code'].nunique():,}")

duplicate_keys = final.duplicated(
    ["transition", "ta_code"]
).sum()

print(f"Duplicate keys    : {duplicate_keys:,}")

counts = (
    final.groupby("transition", observed=False)
    ["ta_code"]
    .nunique()
)

print("\nTAs per transition:")
print(counts.to_string())

# -----------------------------------------------------------------------------
# 8. Missingness QC
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("FEATURE MISSINGNESS")
print("=" * 110)

for c in (
    PRIMARY_FEATURES
    + ROBUSTNESS_FEATURES
):

    nmiss = final[c].isna().sum()
    pct = nmiss / len(final) * 100

    print(
        f"{c:<42} "
        f"missing={nmiss:>3} "
        f"({pct:6.3f}%)"
    )

# -----------------------------------------------------------------------------
# 9. Identify all current-rent unavailable rows
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("CURRENT RENT UNAVAILABLE — EXACT ROWS")
print("=" * 110)

current_missing = final.loc[
    final["rental_current_available"] == 0,
    [
        "transition",
        "anchor_date",
        "eligible_rental_month",
        "ta_code",
        "ta_name"
    ]
]

print(f"\nCount: {len(current_missing):,}")
print(current_missing.to_string(index=False))

# -----------------------------------------------------------------------------
# 10. Identify 12M-change unavailable rows
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("12M RENT CHANGE UNAVAILABLE — EXACT ROWS")
print("=" * 110)

change_missing = final.loc[
    final["rental_change_12m_available"] == 0,
    [
        "transition",
        "anchor_date",
        "eligible_rental_month",
        "ta_code",
        "ta_name"
    ]
]

print(f"\nCount: {len(change_missing):,}")
print(change_missing.to_string(index=False))

# -----------------------------------------------------------------------------
# 11. Chatham audit
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("TA067 — CHATHAM ISLANDS AUDIT")
print("=" * 110)

chatham = final.loc[
    final["ta_code"] == "067"
]

print(
    chatham[
        [
            "transition",
            "eligible_rental_month",
            "rental_geomean_rent_log",
            "rental_geomean_rent_change_12m_log",
            "rental_current_available",
            "rental_change_12m_available"
        ]
    ].to_string(index=False)
)

# -----------------------------------------------------------------------------
# 12. Infinite-value audit
# -----------------------------------------------------------------------------

numeric_feature_cols = (
    PRIMARY_FEATURES
    + ROBUSTNESS_FEATURES
)

inf_count = np.isinf(
    final[numeric_feature_cols]
    .to_numpy(dtype=float)
).sum()

print("\n" + "=" * 110)
print("NUMERIC INTEGRITY")
print("=" * 110)

print(f"\nInfinite feature values: {inf_count:,}")

# -----------------------------------------------------------------------------
# 13. Deterministic ordering
# -----------------------------------------------------------------------------

transition_order = [
    "T1", "T2", "T3", "T4", "T5",
    "T6", "T7", "T8", "T9", "T10"
]

final["transition"] = pd.Categorical(
    final["transition"],
    categories=transition_order,
    ordered=True
)

final = (
    final
    .sort_values(
        ["transition", "ta_code"]
    )
    .reset_index(drop=True)
)

# Convert transition back to string for portable parquet
final["transition"] = final["transition"].astype(str)

# -----------------------------------------------------------------------------
# 14. Save
# -----------------------------------------------------------------------------

final.to_parquet(
    OUT_FILE,
    index=False
)

print("\n" + "=" * 110)
print("SAVE")
print("=" * 110)

print(f"\nSaved to:\n{OUT_FILE}")

# -----------------------------------------------------------------------------
# 15. Exact reload verification
# -----------------------------------------------------------------------------

reload = pd.read_parquet(OUT_FILE)

same_shape = reload.shape == final.shape
same_columns = list(reload.columns) == list(final.columns)

same_keys = reload[
    ["transition", "ta_code"]
].equals(
    final[
        ["transition", "ta_code"]
    ]
)

numeric_equal = np.allclose(
    reload[numeric_feature_cols].to_numpy(dtype=float),
    final[numeric_feature_cols].to_numpy(dtype=float),
    equal_nan=True
)

technical_equal = reload[
    TECHNICAL_FIELDS
].equals(
    final[
        TECHNICAL_FIELDS
    ]
)

print("\nRELOAD VERIFICATION")
print("-" * 110)

print(f"Same shape       : {same_shape}")
print(f"Same columns     : {same_columns}")
print(f"Same keys        : {same_keys}")
print(f"Numeric exact    : {numeric_equal}")
print(f"Technical exact  : {technical_equal}")

# -----------------------------------------------------------------------------
# 16. Final decision
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("STEP 22.9 FINAL STATUS")
print("=" * 110)

pass_conditions = (
    len(final) == 670
    and final["transition"].nunique() == 10
    and final["ta_code"].nunique() == 67
    and duplicate_keys == 0
    and inf_count == 0
    and len(chatham) == 10
    and chatham["rental_current_available"].sum() == 0
    and same_shape
    and same_columns
    and same_keys
    and numeric_equal
    and technical_equal
)

if pass_conditions:

    print("STEP 22.9 STATUS: RENTAL-MARKET FEATURE TABLE FROZEN")

else:

    print("STEP 22.9 STATUS: REVIEW REQUIRED")

print("=" * 110)

In [ ]:
# =============================================================================
# STEP 22.10 — MCERT PROPERTY & SALES RAW WORKBOOK STRUCTURE AUDIT
#
# Purpose:
#   Inspect the raw MCERT Property & Sales workbook before any processing.
#
# We will check:
#   1. File existence
#   2. Workbook sheets
#   3. Dimensions of every sheet
#   4. Raw first rows (without assuming headers)
#   5. Candidate date/geography/value fields
#   6. Missingness and duplicate-column names
#
# IMPORTANT:
#   - RAW AUDIT ONLY
#   - Do not modify raw workbook
#   - No feature engineering
#   - No temporal alignment
#   - No TA crosswalk yet
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 110)
print("STEP 22.10 — MCERT PROPERTY & SALES RAW WORKBOOK STRUCTURE AUDIT")
print("=" * 110)

# -----------------------------------------------------------------------------
# 1. Locate workbook
# -----------------------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI"
)

FILE = (
    PROJECT_ROOT
    / "01_raw"
    / "property_market"
    / "Property-and-Sales-data-download-June-2026.xlsx"
)

print("\nFILE")
print("-" * 110)
print("Path   :", FILE)
print("Exists :", FILE.exists())

if not FILE.exists():
    raise FileNotFoundError(FILE)

print(
    "Size MB:",
    round(FILE.stat().st_size / (1024 ** 2), 3)
)

# -----------------------------------------------------------------------------
# 2. Workbook inventory
# -----------------------------------------------------------------------------

xls = pd.ExcelFile(FILE)

print("\n" + "=" * 110)
print("WORKBOOK SHEETS")
print("=" * 110)

print(f"\nNumber of sheets: {len(xls.sheet_names)}")

for i, sheet in enumerate(xls.sheet_names, start=1):
    print(f"{i:>2}. {sheet}")

# -----------------------------------------------------------------------------
# 3. Read every sheet WITHOUT assuming header structure
# -----------------------------------------------------------------------------

sheet_inventory = []

raw_sheets = {}

for sheet in xls.sheet_names:

    tmp = pd.read_excel(
        FILE,
        sheet_name=sheet,
        header=None
    )

    raw_sheets[sheet] = tmp

    sheet_inventory.append({
        "sheet": sheet,
        "rows": tmp.shape[0],
        "columns": tmp.shape[1],
        "nonempty_cells": int(tmp.notna().sum().sum()),
        "completely_empty_rows": int(tmp.isna().all(axis=1).sum()),
        "completely_empty_columns": int(tmp.isna().all(axis=0).sum())
    })

sheet_inventory = pd.DataFrame(sheet_inventory)

print("\n" + "=" * 110)
print("SHEET DIMENSIONS")
print("=" * 110)

print(
    sheet_inventory.to_string(index=False)
)

# -----------------------------------------------------------------------------
# 4. Show first 15 rows of each sheet
#
# This lets us identify:
#   - true header row
#   - metadata rows
#   - geography layout
#   - time layout
#   - measure layout
# -----------------------------------------------------------------------------

for sheet, tmp in raw_sheets.items():

    print("\n\n" + "=" * 110)
    print(f"SHEET PREVIEW: {sheet}")
    print("=" * 110)

    pd.set_option("display.max_columns", 20)
    pd.set_option("display.width", 220)
    pd.set_option("display.max_colwidth", 50)

    print(
        tmp.head(15).to_string(
            index=True,
            header=False
        )
    )

# -----------------------------------------------------------------------------
# 5. Search raw workbook for important keywords
# -----------------------------------------------------------------------------

keywords = [
    "territorial",
    "authority",
    "region",
    "date",
    "month",
    "quarter",
    "year",
    "price",
    "index",
    "hpi",
    "sales",
    "median",
    "mean"
]

keyword_hits = []

for sheet, tmp in raw_sheets.items():

    for row_idx in range(len(tmp)):

        row_values = (
            tmp.iloc[row_idx]
            .dropna()
            .astype(str)
            .tolist()
        )

        row_text = " | ".join(row_values)
        row_text_lower = row_text.lower()

        matched = [
            k for k in keywords
            if k in row_text_lower
        ]

        if matched:

            keyword_hits.append({
                "sheet": sheet,
                "excel_row": row_idx + 1,
                "keywords": ", ".join(matched),
                "row_preview": row_text[:500]
            })

keyword_hits = pd.DataFrame(keyword_hits)

print("\n" + "=" * 110)
print("KEYWORD SEARCH")
print("=" * 110)

if len(keyword_hits) == 0:

    print("\nNo keyword hits found.")

else:

    print(f"\nKeyword-hit rows: {len(keyword_hits):,}")

    print(
        keyword_hits.head(100).to_string(
            index=False
        )
    )

    if len(keyword_hits) > 100:
        print(
            f"\nNOTE: Showing first 100 of "
            f"{len(keyword_hits):,} keyword-hit rows."
        )

# -----------------------------------------------------------------------------
# 6. Attempt normal header read for each sheet
#
# QC only — we are NOT assuming this is correct yet.
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("DEFAULT-HEADER READ — QC ONLY")
print("=" * 110)

for sheet in xls.sheet_names:

    tmp = pd.read_excel(
        FILE,
        sheet_name=sheet
    )

    print("\n" + "-" * 110)
    print(f"SHEET: {sheet}")
    print("-" * 110)

    print(
        f"Shape under default header assumption: "
        f"{tmp.shape}"
    )

    print("\nColumns:")

    for i, col in enumerate(tmp.columns, start=1):
        print(
            f"{i:>3}. {repr(col)}"
        )

    print("\nFirst 5 rows:")

    print(
        tmp.head(5).to_string(
            index=False
        )
    )

# -----------------------------------------------------------------------------
# 7. Check duplicate column names under default header interpretation
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("DUPLICATE COLUMN-NAME AUDIT")
print("=" * 110)

for sheet in xls.sheet_names:

    tmp = pd.read_excel(
        FILE,
        sheet_name=sheet
    )

    duplicate_cols = (
        pd.Index(tmp.columns)
        .duplicated()
        .sum()
    )

    print(
        f"{sheet:<35} "
        f"duplicate columns = {duplicate_cols}"
    )

# -----------------------------------------------------------------------------
# 8. Final summary
# -----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("STEP 22.10 SUMMARY")
print("=" * 110)

print(f"""
Workbook exists : {FILE.exists()}
Sheets          : {len(xls.sheet_names)}

No feature engineering has been performed.
No HPI values have been selected.
No geography crosswalk has been performed.
No point-in-time assumptions have been made.
""")

print(
    "STEP 22.10 STATUS: RAW WORKBOOK STRUCTURE AUDITED — "
    "REVIEW OUTPUT BEFORE PROCESSING"
)

print("=" * 110)

In [ ]:
# =============================================================================
# STEP 22.11 — MCERT HOUSE PRICE INDEX STRUCTURE / COVERAGE AUDIT
# =============================================================================

import pandas as pd
import numpy as np

print("=" * 110)
print("STEP 22.11 — MCERT HOUSE PRICE INDEX STRUCTURE / COVERAGE AUDIT")
print("=" * 110)

# -------------------------------------------------------------------------
# 1. Read HPI sheet only
# -------------------------------------------------------------------------

hpi = pd.read_excel(
    FILE,
    sheet_name="House Price Index"
)

hpi["period"] = pd.to_datetime(
    hpi["period"],
    errors="coerce"
)

print("\nBASIC STRUCTURE")
print("-" * 110)

print("Rows              :", f"{len(hpi):,}")
print("Columns           :", hpi.shape[1])
print("Unparsed periods  :", hpi["period"].isna().sum())
print("Min period        :", hpi["period"].min())
print("Max period        :", hpi["period"].max())

# -------------------------------------------------------------------------
# 2. Dimension categories
# -------------------------------------------------------------------------

print("\n" + "=" * 110)
print("STAT_TYPE")
print("=" * 110)

print(
    hpi["stat_type"]
    .value_counts(dropna=False)
    .to_string()
)

print("\n" + "=" * 110)
print("AREA_TYPE")
print("=" * 110)

print(
    hpi["area_type"]
    .value_counts(dropna=False)
    .to_string()
)

print("\n" + "=" * 110)
print("VALUE_QUARTILE")
print("=" * 110)

print(
    hpi["value_quartile"]
    .value_counts(dropna=False)
    .to_string()
)

# -------------------------------------------------------------------------
# 3. Geography inventory
# -------------------------------------------------------------------------

geo = (
    hpi[
        ["area_type", "area"]
    ]
    .drop_duplicates()
    .sort_values(
        ["area_type", "area"]
    )
)

print("\n" + "=" * 110)
print("GEOGRAPHY INVENTORY")
print("=" * 110)

geo_counts = (
    geo.groupby("area_type")
    .size()
    .rename("unique_areas")
)

print("\nUnique areas by area_type:")
print(geo_counts.to_string())

# Territorial Authorities specifically
ta_hpi = hpi.loc[
    hpi["area_type"].eq(
        "Territorial Authority"
    )
].copy()

ta_names = sorted(
    ta_hpi["area"]
    .dropna()
    .unique()
)

print("\nTerritorial Authority names:")
print("Count:", len(ta_names))

for x in ta_names:
    print(" -", x)

# -------------------------------------------------------------------------
# 4. TA HPI dimensional structure
# -------------------------------------------------------------------------

print("\n" + "=" * 110)
print("TA HPI DIMENSION COMBINATIONS")
print("=" * 110)

ta_dimensions = (
    ta_hpi.groupby(
        ["stat_type", "value_quartile"],
        dropna=False
    )
    .size()
    .reset_index(name="rows")
)

print(
    ta_dimensions.to_string(
        index=False
    )
)

# -------------------------------------------------------------------------
# 5. Temporal coverage by TA / stat type / quartile
# -------------------------------------------------------------------------

coverage = (
    ta_hpi.groupby(
        ["area", "stat_type", "value_quartile"],
        dropna=False
    )
    .agg(
        n_rows=("period", "size"),
        min_period=("period", "min"),
        max_period=("period", "max"),
        hpi_available=(
            "house_price_index",
            lambda x: x.notna().sum()
        ),
        annual_change_available=(
            "annual_change_in_hpi_percent",
            lambda x: x.notna().sum()
        )
    )
    .reset_index()
)

print("\n" + "=" * 110)
print("TA TEMPORAL COVERAGE SUMMARY")
print("=" * 110)

print("Coverage groups:", f"{len(coverage):,}")

print("\nEarliest period:")
print(coverage["min_period"].min())

print("\nLatest period:")
print(coverage["max_period"].max())

print("\nRows per TA/dimension combination:")
print(
    coverage["n_rows"]
    .describe()
    .to_string()
)

# -------------------------------------------------------------------------
# 6. Recent monthly ALL-quartile TA subset
#
# QC only. This is NOT yet a feature selection decision.
# -------------------------------------------------------------------------

recent = ta_hpi.loc[
    (ta_hpi["stat_type"] == "Monthly")
    & (ta_hpi["value_quartile"] == "All")
    & (ta_hpi["period"] >= "2024-01-01")
].copy()

print("\n" + "=" * 110)
print("RECENT MONTHLY / ALL-QUARTILE TA HPI")
print("=" * 110)

print("Rows          :", f"{len(recent):,}")
print("Unique TAs    :", recent["area"].nunique())
print("Unique months :", recent["period"].nunique())
print("Min month     :", recent["period"].min())
print("Max month     :", recent["period"].max())

print("\nMissing values:")
for c in [
    "residential_sales_used_for_HPI",
    "house_price_index",
    "annual_change_in_hpi_percent"
]:
    print(
        f"{c:<35}",
        recent[c].isna().sum()
    )

# -------------------------------------------------------------------------
# 7. TA coverage by recent month
# -------------------------------------------------------------------------

monthly_coverage = (
    recent.groupby("period")
    .agg(
        n_tas=("area", "nunique"),
        hpi_available=(
            "house_price_index",
            lambda x: x.notna().sum()
        ),
        annual_change_available=(
            "annual_change_in_hpi_percent",
            lambda x: x.notna().sum()
        )
    )
    .reset_index()
)

print("\n" + "=" * 110)
print("RECENT COVERAGE BY MONTH")
print("=" * 110)

print(
    monthly_coverage.to_string(
        index=False
    )
)

# -------------------------------------------------------------------------
# 8. Duplicate key audit
# -------------------------------------------------------------------------

dup = recent.duplicated(
    subset=[
        "area",
        "period",
        "stat_type",
        "value_quartile"
    ],
    keep=False
)

print("\n" + "=" * 110)
print("KEY UNIQUENESS")
print("=" * 110)

print(
    "Rows involved in duplicate "
    "TA × month × stat_type × quartile keys:",
    dup.sum()
)

# -------------------------------------------------------------------------
# 9. Numeric sanity
# -------------------------------------------------------------------------

print("\n" + "=" * 110)
print("NUMERIC SANITY — RECENT MONTHLY ALL")
print("=" * 110)

for c in [
    "residential_sales_used_for_HPI",
    "house_price_index",
    "annual_change_in_hpi_percent"
]:

    s = recent[c]

    print(f"\n{c}")
    print("  available :", s.notna().sum())
    print("  missing   :", s.isna().sum())
    print("  min       :", s.min())
    print("  median    :", s.median())
    print("  max       :", s.max())

# -------------------------------------------------------------------------
# 10. Final summary
# -------------------------------------------------------------------------

print("\n" + "=" * 110)
print("STEP 22.11 SUMMARY")
print("=" * 110)

print(f"""
Total HPI rows                    : {len(hpi):,}
TA HPI rows                       : {len(ta_hpi):,}
Unique TA names                   : {ta_hpi['area'].nunique():,}

Recent Monthly / All rows         : {len(recent):,}
Recent unique TAs                 : {recent['area'].nunique():,}
Recent unique months              : {recent['period'].nunique():,}

Duplicate recent keys             : {dup.sum():,}
""")

print(
    "STEP 22.11 STATUS: HPI STRUCTURE AND COVERAGE AUDITED — "
    "NOT YET FEATURED"
)

print("=" * 110)

In [ ]:
# =============================================================================
# STEP 22.12 — FREEZE MCERT HPI METHODOLOGICAL ROLE
# =============================================================================

from pathlib import Path
import pandas as pd

print("=" * 110)
print("STEP 22.12 — FREEZE MCERT HPI METHODOLOGICAL ROLE")
print("=" * 110)

PROJECT_ROOT = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")

OUT_DIR = PROJECT_ROOT / "07_provenance"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / "mcert_hpi_methodological_decision.csv"

decision = pd.DataFrame([{
    "source":
        "MCERT Property and Sales Statistics - House Price Index",

    "workbook":
        "Property-and-Sales-data-download-June-2026.xlsx",

    "vintage":
        "June 2026",

    "geography":
        "Territorial Authority",

    "total_ta_names_in_hpi_sheet":
        int(ta_hpi["area"].nunique()),

    "recent_monthly_all_tas":
        int(recent["area"].nunique()),

    "recent_period_start":
        recent["period"].min().date(),

    "recent_period_end":
        recent["period"].max().date(),

    "recent_monthly_all_rows":
        len(recent),

    "annual_change_missing_n":
        int(recent["annual_change_in_hpi_percent"].isna().sum()),

    "series_revision_caveat":
        "Entire series is revised each month as additional sales records are received.",

    "reporting_lag_caveat":
        "Sales data are reported after approximately a three-month lag.",

    "primary_predictor_eligible":
        False,

    "point_in_time_t1_t10_eligible":
        False,

    "frozen_role":
        "Retrospective robustness/context only",

    "reason":
        (
            "The June-2026 workbook is a revised historical vintage and does not "
            "provide historical release vintages required to reconstruct the exact "
            "information set available at each T1-T10 prediction origin. Recent "
            "Monthly-All TA HPI coverage is also incomplete."
        )
}])

decision.to_csv(
    OUT_FILE,
    index=False
)

print("\nFROZEN DECISION")
print("-" * 110)

print(decision.T.to_string(header=False))

print("\n" + "=" * 110)
print("SAVE")
print("=" * 110)

print("\nSaved to:")
print(OUT_FILE)

# Reload verification
check = pd.read_csv(OUT_FILE)

print("\nReload rows   :", len(check))
print("Reload cols   :", len(check.columns))
print(
    "Frozen role  :",
    check.loc[0, "frozen_role"]
)

assert len(check) == 1
assert check.loc[0, "primary_predictor_eligible"] in [False, "False", 0]
assert check.loc[0, "point_in_time_t1_t10_eligible"] in [False, "False", 0]

print("\n" + "=" * 110)
print("STEP 22.12 STATUS: MCERT HPI ROLE FROZEN")
print("=" * 110)

print("""
PRIMARY MODEL:
    MCERT HPI = EXCLUDED

POINT-IN-TIME T1-T10 FEATURES:
    MCERT HPI = EXCLUDED

RETROSPECTIVE ROBUSTNESS / CONTEXT:
    MCERT HPI = RETAINED AS OPTIONAL SOURCE

No HPI feature table has been created.
No pseudo-lagged HPI has been created.
""")